In [ ]:

from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

from langgraph.graph import StateGraph,START,END

In [26]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-32B",
    temperature=0.7,
    max_new_tokens=1000
)

model= ChatHuggingFace(llm=llm)

In [27]:
class joke_state(TypedDict):
    topic :str
    joke: str
    expln: str 

In [28]:
def genr_joke(state: joke_state):
    prompt=f'generate a joke on a topic {state['topic']}'
    response=model.invoke(prompt).content

    return {'joke':response}

In [29]:
def joke_expln(state:joke_state):
    prompt=f'write an explanation for the joke{state['joke']}'
    response=model.invoke(prompt).content
    return{'expln':response}


In [30]:
graph=StateGraph(joke_state)
graph.add_node('genr_joke',genr_joke)
graph.add_node('joke_expln',joke_expln)

graph.add_edge(START,'genr_joke')
graph.add_edge('genr_joke','joke_expln')
graph.add_edge('joke_expln',END)

checkpointer=InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)

In [31]:
config1={'configurable':{"thread_id":'1'}}
workflow.invoke({'topic':'pizza'},config=config1)

{'topic': 'pizza',
 'joke': '',
 'expln': "\n\nIt seems I don't have the specific joke to explain. Could you please share the joke you'd like me to explain? Once you provide it, I’ll break down the humor, wordplay, or context behind it! 😊"}

In [32]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '', 'expln': "\n\nIt seems I don't have the specific joke to explain. Could you please share the joke you'd like me to explain? Once you provide it, I’ll break down the humor, wordplay, or context behind it! 😊"}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-f04e-6a8c-8002-0f874a3833bf'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-03T06:35:54.282844+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-c84b-6b66-8001-8a6ed15d164f'}}, tasks=(), interrupts=())

In [33]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '', 'expln': "\n\nIt seems I don't have the specific joke to explain. Could you please share the joke you'd like me to explain? Once you provide it, I’ll break down the humor, wordplay, or context behind it! 😊"}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-f04e-6a8c-8002-0f874a3833bf'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-03T06:35:54.282844+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-c84b-6b66-8001-8a6ed15d164f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': ''}, next=('joke_expln',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-c84b-6b66-8001-8a6ed15d164f'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-03T06:35:50.087334+00:00', parent_config={'configurable': {'thread_id': '1

In [34]:
config2={'configurable':{'thread_id':2}}
workflow.invoke({'topic':'potato'},config=config2)

{'topic': 'potato',
 'joke': '',
 'expln': "\n\nSure! However, I need to know **which joke** you'd like me to explain. Could you please share the specific joke you're referring to? Once I have the text, I can break down its humor, wordplay, or context for you. 😊"}

In [35]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '', 'expln': "\n\nIt seems I don't have the specific joke to explain. Could you please share the joke you'd like me to explain? Once you provide it, I’ll break down the humor, wordplay, or context behind it! 😊"}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-f04e-6a8c-8002-0f874a3833bf'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-03T06:35:54.282844+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-c84b-6b66-8001-8a6ed15d164f'}}, tasks=(), interrupts=())

In [36]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '', 'expln': "\n\nIt seems I don't have the specific joke to explain. Could you please share the joke you'd like me to explain? Once you provide it, I’ll break down the humor, wordplay, or context behind it! 😊"}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-f04e-6a8c-8002-0f874a3833bf'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-03T06:35:54.282844+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-c84b-6b66-8001-8a6ed15d164f'}}, tasks=(), interrupts=())

Time Travel

In [37]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": '1f1beef8-4976-6ec1-8009-dfb35531807f'}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1beef8-4976-6ec1-8009-dfb35531807f'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [40]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id":'1f1beef8-4976-6ec1-8009-dfb35531807f'}})

EmptyInputError: Received no input for __start__

In [41]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '', 'expln': "\n\nIt seems I don't have the specific joke to explain. Could you please share the joke you'd like me to explain? Once you provide it, I’ll break down the humor, wordplay, or context behind it! 😊"}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-f04e-6a8c-8002-0f874a3833bf'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-03T06:35:54.282844+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-c84b-6b66-8001-8a6ed15d164f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': ''}, next=('joke_expln',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef4a-c84b-6b66-8001-8a6ed15d164f'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-03T06:35:50.087334+00:00', parent_config={'configurable': {'thread_id': '1

Updating State

In [43]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1beef8-4976-6ec1-8009-dfb35531807f", "checkpoint_ns": ""}}, {'topic':'potato'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1bef54-88a6-6c56-8000-00adad5572da'}}

In [44]:
list(workflow.get_state_history(config1))


[StateSnapshot(values={'topic': 'potato'}, next=('genr_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef54-88a6-6c56-8000-00adad5572da'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-10-03T06:40:11.849207+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1beef8-4976-6ec1-8009-dfb35531807f'}}, tasks=(PregelTask(id='5c5de35c-986b-1297-55b4-9cd87ba8e4b5', name='genr_joke', path=('__pregel_pull', 'genr_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'potato'}, next=('genr_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1bef52-aed4-6f52-8000-39ebaa19f450'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-10-03T06:39:22.165626+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-723

In [46]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1bef4a-15f9-60da-bfff-0b88c8faaa42"}})

{'topic': 'pizza',
 'joke': '',
 'expln': '\n\nCertainly! Let’s walk through a classic joke and break down why it’s funny. Here’s an example:\n\n**Joke:**  \n"Why don\'t scientists trust atoms?  \nBecause they make up everything!"\n\n---\n\n### **Explanation:**\n\n1. **The Setup:**  \n   The question "Why don\'t scientists trust atoms?" primes the listener to think about the role of atoms in science. Atoms are the basic building blocks of matter, so the expectation is that the answer will relate to their scientific properties.\n\n2. **The Punchline:**  \n   The answer "Because they make up everything!" subverts expectations. It plays on two meanings of the phrase **"make up":**  \n   - **Literal meaning:** Atoms are the physical components of all matter (they "make up" everything).  \n   - **Figurative meaning:** To fabricate or lie ("make up a story").  \n\n3. **Wordplay & Surprise:**  \n   The humor arises from the **pun**—a play on words with double meanings. The listener initially 

In [47]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '', 'expln': '\n\nCertainly! Let’s walk through a classic joke and break down why it’s funny. Here’s an example:\n\n**Joke:**  \n"Why don\'t scientists trust atoms?  \nBecause they make up everything!"\n\n---\n\n### **Explanation:**\n\n1. **The Setup:**  \n   The question "Why don\'t scientists trust atoms?" primes the listener to think about the role of atoms in science. Atoms are the basic building blocks of matter, so the expectation is that the answer will relate to their scientific properties.\n\n2. **The Punchline:**  \n   The answer "Because they make up everything!" subverts expectations. It plays on two meanings of the phrase **"make up":**  \n   - **Literal meaning:** Atoms are the physical components of all matter (they "make up" everything).  \n   - **Figurative meaning:** To fabricate or lie ("make up a story").  \n\n3. **Wordplay & Surprise:**  \n   The humor arises from the **pun**—a play on words with double meanings. The

fault Tolerance

In [48]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [49]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [50]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [51]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...


AttributeError: 'StateGraph' object has no attribute 'invoke'

In [52]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


AttributeError: 'StateGraph' object has no attribute 'invoke'

InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver

This is extremely important for this notebook.

Its job is:

Save/checkpoint the state of the graph in memory.

For example:

User input
   ↓
Generate joke
   ↓
Explain joke
   ↓
Save state

Later we can retrieve that saved state.

This is what enables:

Persistence
State history
Time travel
Resume
Updating previous states
Fault tolerance


get_state()
workflow.get_state(config1)
Function

Get the current saved state of a particular thread.

Because:

config1

contains:

thread_id = "1"

LangGraph retrieves the state belonging to thread 1.

You can think:

"Hey LangGraph, show me what you currently remember about thread 1."

get_state_history()
workflow.get_state_history(config1)

This is even more interesting.

It gives you the history of checkpoints.

For example:

Checkpoint 1
    ↓
Checkpoint 2
    ↓
Checkpoint 3
    ↓
Checkpoint 4

To see them as a list:

list(workflow.get_state_history(config1))